# Stage 3 Qwen One-Pass Clean (Kaggle)

Leakage-free GT-crop rerun notebook. It uses a `_nocroppath` prompt version, runs preflight + full val + eval + visuals, and packs the full artifact set into one archive.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]

JSONL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

BACKEND_MODE = "qwen_hf"
PROMPT_VERSION = "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath"
RUN_ID = "stage3_qwen_val_v2_clean_final"

DO_PREFLIGHT = True
PREFLIGHT_SAMPLES = 1
DO_FULL_RUN = True
MAX_VISIBILITY_ERROR_SAMPLES = 18

print("RUN_ID:", RUN_ID)


In [ ]:
def sh(cmd: str, cwd: Path | None = None, check: bool = True):
    print(f"$ {cmd}")
    p = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p

def path_exists_or_raise(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


In [ ]:
# 1) Setup: clean clone + hard clean-prompt checks + deps + GPU check
import yaml

if REPO_DIR.exists():
    print("Removing existing repo to avoid stale prompts:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")

git_head = subprocess.check_output("git rev-parse --short HEAD", shell=True, cwd=str(REPO_DIR), text=True).strip()
print("git_head:", git_head)

cfg_path = REPO_DIR / "configs/pipeline/stage3_vlm_gt_baseline.yaml"
if not cfg_path.exists():
    raise FileNotFoundError(f"Missing required repo file: {cfg_path}")
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
versions = cfg["prompt"]["versions"]
if PROMPT_VERSION not in versions:
    raise RuntimeError(f"Prompt version not found in config: {PROMPT_VERSION}")
if not PROMPT_VERSION.endswith("_nocroppath"):
    raise RuntimeError("Selected prompt version is not a clean _nocroppath variant")
prompt_system_rel = Path(versions[PROMPT_VERSION]["system_path"])
prompt_user_rel = Path(versions[PROMPT_VERSION]["user_path"])
for rel in [prompt_system_rel, prompt_user_rel]:
    abs_path = REPO_DIR / rel
    if not abs_path.exists():
        raise FileNotFoundError(f"Missing required repo file: {abs_path}")

sys_text = (REPO_DIR / prompt_system_rel).read_text(encoding="utf-8")
usr_text = (REPO_DIR / prompt_user_rel).read_text(encoding="utf-8")
if "Visibility policy:" not in sys_text:
    raise RuntimeError("Unexpected system prompt content")
if "crop_path" in usr_text:
    raise RuntimeError("Selected clean user prompt still exposes crop_path")

print("Resolved system prompt:", prompt_system_rel)
print("Resolved user prompt  :", prompt_user_rel)
print("Repo clean-prompt checks: OK")

sh("python -m pip install -q -U pip")
sh(f"python -m pip install -q -r {REPO_DIR / 'requirements.txt'}")
sh("python -m pip install -q -U transformers accelerate qwen-vl-utils")

sh("nvidia-smi", check=False)
print("cwd:", REPO_DIR)


In [ ]:
# 2) Resolve dataset jsonl path + root
VAL_JSONL = None
DATASET_ROOT = None

for root in DATASET_ROOT_CANDIDATES:
    p = root / JSONL_REL
    if p.exists():
        VAL_JSONL = p
        DATASET_ROOT = root
        break

if VAL_JSONL is None:
    found = list(Path("/kaggle/input").rglob("vlm_labels_v1_val_v2.annotated.jsonl"))
    if found:
        VAL_JSONL = found[0]
        # expected .../<dataset_root>/stage3_regrouped_v2/val/file.jsonl
        DATASET_ROOT = VAL_JSONL.parents[2]

if VAL_JSONL is None:
    raise FileNotFoundError("Could not locate vlm_labels_v1_val_v2.annotated.jsonl under /kaggle/input")

print("VAL_JSONL:", VAL_JSONL)
print("DATASET_ROOT:", DATASET_ROOT)


In [ ]:
# 3) Preflight run (cheap sanity)
if DO_PREFLIGHT:
    preflight_run_id = f"{RUN_ID}_preflight"
    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "run_stage3_vlm_baseline.py"),
            "--config", str(REPO_DIR / "configs" / "pipeline" / "stage3_vlm_gt_baseline.yaml"),
            "--backend-mode", BACKEND_MODE,
            "--prompt-version", PROMPT_VERSION,
            "--input-jsonl", f'"{VAL_JSONL}"',
            "--run-id", preflight_run_id,
            "--max-samples", str(PREFLIGHT_SAMPLES),
            "--no-resume",
        ]),
        cwd=REPO_DIR,
    )
    print("Preflight done:", preflight_run_id)
else:
    print("Preflight skipped")


In [ ]:
# 4) Full run + validate + eval + visuals
if DO_FULL_RUN:
    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "run_stage3_vlm_baseline.py"),
            "--config", str(REPO_DIR / "configs" / "pipeline" / "stage3_vlm_gt_baseline.yaml"),
            "--backend-mode", BACKEND_MODE,
            "--prompt-version", PROMPT_VERSION,
            "--input-jsonl", f'"{VAL_JSONL}"',
            "--run-id", RUN_ID,
            "--no-resume",
        ]),
        cwd=REPO_DIR,
    )

    RUN_DIR = REPO_DIR / "outputs" / "stage3_vlm_baseline_runs" / RUN_ID
    EVAL_DIR = RUN_DIR / "eval"
    PRED_JSONL = RUN_DIR / "predictions_vlm_labels_v1.jsonl"

    path_exists_or_raise(RUN_DIR, "RUN_DIR")
    path_exists_or_raise(PRED_JSONL, "predictions_vlm_labels_v1.jsonl")

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "validate_vlm_labels_v1.py"),
            "--input", f'"{PRED_JSONL}"',
        ]),
        cwd=REPO_DIR,
    )

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "eval_stage3_vlm_baseline.py"),
            "--run-dir", f'"{RUN_DIR}"',
            "--ground-truth-jsonl", f'"{VAL_JSONL}"',
        ]),
        cwd=REPO_DIR,
    )

    sh(
        " ".join([
            "python",
            str(REPO_DIR / "scripts" / "visualize_stage3_eval_results.py"),
            "--eval-dir", f'"{EVAL_DIR}"',
        ]),
        cwd=REPO_DIR,
    )

    print("Full run done:", RUN_ID)
    print("RUN_DIR:", RUN_DIR)
else:
    raise RuntimeError("DO_FULL_RUN=False: notebook expects full run for final artifacts")


In [ ]:
# 5) Print key metrics in notebook output
import pandas as pd

METRICS_PATH = EVAL_DIR / "metrics.json"
RUN_SUMMARY_PATH = RUN_DIR / "run_summary.json"

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
run_summary = json.loads(RUN_SUMMARY_PATH.read_text(encoding="utf-8"))

rates = metrics.get("rates", {})
f1 = metrics.get("f1", {})
counts = metrics.get("counts", {})
prompt_flags = run_summary.get("prompt_selection", {})

summary = {
    "run_id": RUN_ID,
    "prompt_version": PROMPT_VERSION,
    "backend": run_summary.get("backend_mode_effective"),
    "model": run_summary.get("backend_settings_effective", {}).get("model"),
    "user_prompt_contains_crop_path_token": prompt_flags.get("user_prompt_contains_crop_path_token"),
    "records_attempted": run_summary.get("counters", {}).get("records_attempted"),
    "status_ok": run_summary.get("counters", {}).get("status_ok"),
    "status_backend_error": run_summary.get("counters", {}).get("status_backend_error"),
    "parse_success_rate": rates.get("parse_success_rate"),
    "schema_valid_rate": rates.get("schema_valid_rate"),
    "coarse_class_accuracy": rates.get("coarse_class_accuracy"),
    "coarse_class_macro_f1": f1.get("coarse_class_macro_f1"),
    "visibility_accuracy": rates.get("visibility_accuracy"),
    "visibility_macro_f1": f1.get("visibility_macro_f1"),
    "needs_review_accuracy": rates.get("needs_review_accuracy"),
    "tag_exact_match_rate": rates.get("tag_exact_match_rate"),
    "tag_mean_jaccard": rates.get("tag_mean_jaccard"),
    "pred_ambiguous_rate": rates.get("pred_ambiguous_rate"),
    "gt_ambiguous_rate": rates.get("gt_ambiguous_rate"),
    "samples_evaluated": counts.get("samples_evaluated") or counts.get("evaluated_total"),
}

print("=== STAGE3 CLEAN RUN SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nmetrics.json:", METRICS_PATH)
print("run_summary.json:", RUN_SUMMARY_PATH)

summary_df = pd.DataFrame([summary]).T
summary_df.columns = ["value"]
summary_df


In [ ]:
# 6) Visual analysis in notebook (KPI + confusion)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Markdown

display(Markdown("## Visual Analysis"))

visuals_dir = EVAL_DIR / "visuals"
vis_paths = [
    visuals_dir / "kpi_overview.png",
    visuals_dir / "confusion_coarse_class.png",
    visuals_dir / "confusion_visibility.png",
]

existing = [p for p in vis_paths if p.exists()]
if existing:
    fig, axes = plt.subplots(len(existing), 1, figsize=(10, 5 * len(existing)))
    if len(existing) == 1:
        axes = [axes]
    for ax, path in zip(axes, existing):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(path.name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No visual PNG files found in:", visuals_dir)

conf_vis_csv = EVAL_DIR / "confusion_visibility.csv"
if conf_vis_csv.exists():
    conf_vis_df = pd.read_csv(conf_vis_csv)
    display(Markdown("### Visibility Confusion (CSV)"))
    display(conf_vis_df)


In [ ]:
# 7) Visibility error gallery (show real crop images for mismatches)
import math
from PIL import Image

review_csv = EVAL_DIR / "review_table.csv"
if not review_csv.exists():
    raise FileNotFoundError(f"review_table.csv not found: {review_csv}")

review_df = pd.read_csv(review_csv)
required_cols = ["record_id", "crop_path", "gt_visibility", "pred_visibility", "gt_coarse_class", "pred_coarse_class"]
for col in required_cols:
    if col not in review_df.columns:
        raise KeyError(f"Missing column in review_table.csv: {col}")

vis_err_df = review_df[
    review_df["gt_visibility"].notna()
    & review_df["pred_visibility"].notna()
    & (review_df["gt_visibility"] != review_df["pred_visibility"])
].copy()

vis_err_df = vis_err_df.sort_values(by=["gt_visibility", "pred_visibility", "record_id"]).reset_index(drop=True)
top_vis_err_df = vis_err_df.head(MAX_VISIBILITY_ERROR_SAMPLES).copy()

visuals_dir = EVAL_DIR / "visuals"
visuals_dir.mkdir(parents=True, exist_ok=True)
vis_err_csv_out = visuals_dir / "visibility_errors_top.csv"
top_vis_err_df.to_csv(vis_err_csv_out, index=False)

print(f"visibility mismatches total: {len(vis_err_df)}")
print(f"saved top visibility errors csv: {vis_err_csv_out}")

if len(top_vis_err_df) == 0:
    print("No visibility mismatches to display.")
else:
    def resolve_crop_path(rel_path: str) -> Path | None:
        rp = Path(str(rel_path))
        candidates = [
            DATASET_ROOT / rp if DATASET_ROOT else None,
            REPO_DIR / rp,
            Path('/kaggle/input') / rp,
        ]
        for c in candidates:
            if c is not None and c.exists():
                return c
        return None

    cols = 3
    n = len(top_vis_err_df)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]

    for i in range(rows * cols):
        r = i // cols
        c = i % cols
        ax = axes[r][c]
        ax.axis("off")

        if i >= n:
            continue

        row = top_vis_err_df.iloc[i]
        crop_abs = resolve_crop_path(row["crop_path"])
        title = (
            f"{row['record_id']}\n"
            f"vis: {row['gt_visibility']} -> {row['pred_visibility']}\n"
            f"class: {row['gt_coarse_class']} -> {row['pred_coarse_class']}"
        )

        if crop_abs is None:
            ax.text(0.5, 0.5, f"Image not found\n{row['crop_path']}", ha="center", va="center")
            ax.set_title(title, fontsize=10)
            continue

        try:
            img = Image.open(crop_abs).convert("RGB")
            ax.imshow(img)
            ax.set_title(title, fontsize=10)
        except Exception as exc:
            ax.text(0.5, 0.5, f"Open error: {exc}", ha="center", va="center")
            ax.set_title(title, fontsize=10)

    plt.tight_layout()
    gallery_path = visuals_dir / "visibility_errors_gallery.png"
    plt.savefig(gallery_path, dpi=170, bbox_inches="tight")
    plt.show()
    print("saved gallery:", gallery_path)


In [ ]:
# 8) Collect deliverables + pack archive
DELIVER_DIR = Path(f"/kaggle/working/stage3_deliverables_{RUN_ID}")
if DELIVER_DIR.exists():
    shutil.rmtree(DELIVER_DIR)
DELIVER_DIR.mkdir(parents=True, exist_ok=True)

to_copy = [
    RUN_DIR / "run_summary.json",
    RUN_DIR / "config_snapshot.json",
    RUN_DIR / "predictions_vlm_labels_v1.jsonl",
    RUN_DIR / "parsed_predictions.jsonl",
    RUN_DIR / "raw_responses.jsonl",
    RUN_DIR / "sample_results.jsonl",
    RUN_DIR / "failures.jsonl",
    EVAL_DIR / "metrics.json",
    EVAL_DIR / "confusion_coarse_class.csv",
    EVAL_DIR / "confusion_visibility.csv",
    EVAL_DIR / "review_table.csv",
    EVAL_DIR / "failures.jsonl",
    EVAL_DIR / "visuals" / "report.md",
    EVAL_DIR / "visuals" / "kpi_overview.png",
    EVAL_DIR / "visuals" / "confusion_coarse_class.png",
    EVAL_DIR / "visuals" / "confusion_visibility.png",
    EVAL_DIR / "visuals" / "visibility_errors_top.csv",
    EVAL_DIR / "visuals" / "visibility_errors_gallery.png",
]

for src_path in to_copy:
    if src_path.exists():
        rel = src_path.relative_to(RUN_DIR)
        dst = DELIVER_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, dst)

summary_md = DELIVER_DIR / "RESULT_SUMMARY.md"
summary_md.write_text(
    "\n".join([
        f"# Stage 3 Clean One-Pass Result: {RUN_ID}",
        "",
        f"- prompt_version: {PROMPT_VERSION}",
        f"- backend: {summary['backend']}",
        f"- model: {summary['model']}",
        f"- user_prompt_contains_crop_path_token: {summary['user_prompt_contains_crop_path_token']}",
        f"- records_attempted: {summary['records_attempted']}",
        f"- status_ok: {summary['status_ok']}",
        f"- status_backend_error: {summary['status_backend_error']}",
        "",
        "## Main metrics",
        f"- parse_success_rate: {summary['parse_success_rate']}",
        f"- schema_valid_rate: {summary['schema_valid_rate']}",
        f"- coarse_class_accuracy: {summary['coarse_class_accuracy']}",
        f"- coarse_class_macro_f1: {summary['coarse_class_macro_f1']}",
        f"- visibility_accuracy: {summary['visibility_accuracy']}",
        f"- visibility_macro_f1: {summary['visibility_macro_f1']}",
        f"- needs_review_accuracy: {summary['needs_review_accuracy']}",
        f"- tag_exact_match_rate: {summary['tag_exact_match_rate']}",
        f"- tag_mean_jaccard: {summary['tag_mean_jaccard']}",
        f"- pred_ambiguous_rate: {summary['pred_ambiguous_rate']}",
        f"- gt_ambiguous_rate: {summary['gt_ambiguous_rate']}",
        "",
        f"Ground truth JSONL: {VAL_JSONL}",
        f"Run dir: {RUN_DIR}",
    ]),
    encoding="utf-8",
)

ARCHIVE_BASE = Path(f"/kaggle/working/stage3_deliverables_{RUN_ID}")
ARCHIVE_PATH = shutil.make_archive(str(ARCHIVE_BASE), "gztar", root_dir=DELIVER_DIR)

print("DELIVER_DIR:", DELIVER_DIR)
print("ARCHIVE_PATH:", ARCHIVE_PATH)
print("\nFiles in deliverables:")
for p in sorted(DELIVER_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(DELIVER_DIR))


## Artifacts

Run outputs: `outputs/stage3_vlm_baseline_runs/stage3_qwen_val_v2_clean_final/`
Packed archive: `/kaggle/working/stage3_deliverables_stage3_qwen_val_v2_clean_final.tar.gz`
